### np.linalg.svd

In [1]:
import math

def get_shape(a):
    if not isinstance(a, list):
        return ()
    if len(a) == 0:
        return (0,)
    inner_shape = get_shape(a[0])
    for i in range(1, len(a)):
        if get_shape(a[i]) != inner_shape:
            raise ValueError(
                f"Jagged array: first row shape {inner_shape}, "
                f"but row {i} shape {get_shape(a[i])}"
            )
    return (len(a),) + inner_shape


def _matmul_2d(A, B):
    m = len(A)
    n = len(B[0])
    p = len(B)
    return [[sum(A[i][k] * B[k][j] for k in range(p)) for j in range(n)] for i in range(m)]


def _transpose(A):
    rows = len(A)
    cols = len(A[0]) if rows > 0 else 0
    return [[A[i][j] for i in range(rows)] for j in range(cols)]


def _identity(n):
    return [[1.0 if i == j else 0.0 for j in range(n)] for i in range(n)]


def _copy_matrix(A):
    return [row[:] for row in A]


def _norm_vec(v):
    return math.sqrt(sum(x * x for x in v))


def _deep_copy_square(A):
    return [row[:] for row in A]


def _mat_col(A, j):
    return [A[i][j] for i in range(len(A))]


def _svd_2d(A):
    """
    Compute SVD of matrix A (shape m x n) using the
    one-sided Jacobi algorithm on A^T @ A.
    Returns U (m x m), S (list of k singular values), Vt (n x n)
    where k = min(m, n).
    """
    m = len(A)
    n = len(A[0])

    # Compute A^T A (n x n)
    At = _transpose(A)
    AtA = _matmul_2d(At, A)

    # Compute eigenvalues/vectors of AtA using Jacobi iterations
    # V will contain the right singular vectors (columns)
    V = _identity(n)
    mat = _deep_copy_square(AtA)

    MAX_ITER = 1000
    tol = 1e-12

    for _ in range(MAX_ITER):
        # Find the largest off-diagonal element
        max_off = 0.0
        p_max, q_max = 0, 1

        for p in range(n):
            for q in range(p + 1, n):
                if abs(mat[p][q]) > max_off:
                    max_off = abs(mat[p][q])
                    p_max, q_max = p, q

        if max_off < tol:
            break

        p, q = p_max, q_max

        # Compute Jacobi rotation angle
        if mat[p][p] == mat[q][q]:
            theta = math.pi / 4
        else:
            theta = 0.5 * math.atan2(2 * mat[p][q], mat[p][p] - mat[q][q])

        c = math.cos(theta)
        s = math.sin(theta)

        # Apply rotation to mat and V
        # J^T mat J
        new_mat = [row[:] for row in mat]
        for i in range(n):
            new_mat[i][p] = c * mat[i][p] + s * mat[i][q]
            new_mat[i][q] = -s * mat[i][p] + c * mat[i][q]

        for j in range(n):
            mat[p][j] = c * new_mat[p][j] + s * new_mat[q][j]
            mat[q][j] = -s * new_mat[p][j] + c * new_mat[q][j]

        for j in range(n):
            mat[p][j] = new_mat[p][j]
            mat[q][j] = new_mat[q][j]

        mat = new_mat
        mat[p][p] = c * new_mat[p][p] + s * new_mat[q][p]
        mat[q][q] = -s * new_mat[p][q] + c * new_mat[q][q]
        mat[p][q] = 0.0
        mat[q][p] = 0.0

        # Apply rotation to V
        for i in range(n):
            v_ip = V[i][p]
            v_iq = V[i][q]
            V[i][p] = c * v_ip + s * v_iq
            V[i][q] = -s * v_ip + c * v_iq

    # Singular values = sqrt of eigenvalues of AtA
    singular_vals = [math.sqrt(max(mat[i][i], 0.0)) for i in range(n)]

    # Sort by descending singular value
    order = sorted(range(n), key=lambda i: -singular_vals[i])
    singular_vals = [singular_vals[i] for i in order]
    V = [[V[row][order[col]] for col in range(n)] for row in range(n)]

    # Compute U columns: u_i = A @ v_i / sigma_i
    Vt = _transpose(V)
    U = []
    k = min(m, n)

    for i in range(k):
        v_col = [V[row][i] for row in range(n)]
        u_col = [sum(A[row][j] * v_col[j] for j in range(n)) for row in range(m)]
        sigma = singular_vals[i]
        if sigma > 1e-10:
            u_col = [x / sigma for x in u_col]
        else:
            u_col = [0.0] * m
        U.append(u_col)

    # Extend U to full m x m orthonormal basis using Gram-Schmidt
    while len(U) < m:
        # Start with a standard basis vector
        candidate = None
        for e_idx in range(m):
            e = [1.0 if i == e_idx else 0.0 for i in range(m)]
            # Orthogonalize against existing U columns
            for col in U:
                dot = sum(e[i] * col[i] for i in range(m))
                e = [e[i] - dot * col[i] for i in range(m)]
            norm_e = _norm_vec(e)
            if norm_e > 1e-10:
                candidate = [x / norm_e for x in e]
                break
        if candidate is None:
            candidate = [0.0] * m
        U.append(candidate)

    # U is currently columns stored as rows of a list
    # Build U as m x m matrix
    U_mat = [[U[col][row] for col in range(m)] for row in range(m)]

    S = singular_vals[:k]

    return U_mat, S, Vt


def np_linalg_svd(a, full_matrices=True, compute_uv=True):
    """
    Pure-Python equivalent of np.linalg.svd.
    Returns (U, S, Vt) by default.
    - U: m x m left singular vectors
    - S: list of k singular values, k = min(m, n)
    - Vt: n x n right singular vectors (transposed)
    Only supports 2-D input for now.
    """
    shape = get_shape(a)
    if len(shape) != 2:
        raise ValueError("Only 2-D matrices are supported")

    m, n = shape

    U, S, Vt = _svd_2d(a)

    if not compute_uv:
        return S

    if not full_matrices:
        k = min(m, n)
        U = [row[:k] for row in U]
        Vt = Vt[:k]

    return U, S, Vt

In [2]:
U, S, Vt = np_linalg_svd([[1, 0], [0, 1]])
print(S)

U, S, Vt = np_linalg_svd([[1, 0], [0, 2]])
print(S)

U, S, Vt = np_linalg_svd([[3, 0], [0, 0]])
print(S)

U, S, Vt = np_linalg_svd([[1, 2], [3, 4]])
print(S)

S_only = np_linalg_svd([[1, 2], [3, 4]], compute_uv=False)
print(S_only)

U, S, Vt = np_linalg_svd([[1, 2, 3], [4, 5, 6]])
print(S)
print(len(U), len(U[0]))
print(len(Vt), len(Vt[0]))

[1.0, 1.0]
[2.0, 1.0]
[3.0, 0.0]
[5.464985704219043, 0.3659661906262582]
[5.464985704219043, 0.3659661906262582]
[9.241140458016726, 2.333104543404626]
2 2
3 3


### test cases 

In [3]:
print(np_linalg_svd([[1, 2], [3]])) 

ValueError: Jagged array: first row shape (2,), but row 1 shape (1,)

In [4]:
print(np_linalg_svd([1, 2, 3])) 

ValueError: Only 2-D matrices are supported

In [5]:
print(np_linalg_svd(42)) 

ValueError: Only 2-D matrices are supported

In [6]:
print(np_linalg_svd([[[1, 2], [3, 4]]])) 

ValueError: Only 2-D matrices are supported